# **🏡 Project: The House Price Engine 📈**

## **🌟 What are we building?**

Welcome to the **House Price Prediction Project**! Pricing real estate is notoriously tricky. It’s a mix of hard facts (square footage) and soft variables (neighborhood vibes). For banks, buyers, and platforms like Zillow, getting this wrong by even 5% can mean leaving tens of thousands of dollars on the table.

In this project, we are stepping into the shoes of a **Data Scientist** to build a machine learning model that takes the guesswork out of property values. We will use `pandas` to wrangle messy raw housing data and build a predictive engine that accurately prices homes based on their actual features.

---

## **🗺️ The Roadmap: How we get it done**

We aren't just throwing data at an algorithm and hoping for the best. We’re building a clean, step-by-step pipeline across four key phases:

* **1. Digging into the Data (EDA) 🔍**
  * We'll use `pandas` to pull in our CSV files, check out what data types we're dealing with, and hunt down missing values. 
  * We'll look at the distribution of house prices to see if luxury homes are skewing our numbers, and find out which variables actually correlate with a higher price tag.

* **2. Cleaning & Feature Engineering 🛠️**
  * Raw data is never perfect. We will use `pandas` methods to fill in missing gaps and drop weird outliers (like a massive mansion sold for dirt cheap).
  * We'll create smarter features that the model can understand—like combining individual porch and deck metrics into a single "Total Outdoor Space" variable, or calculating exactly how old a house was the year it was sold.

* **3. Training the Models 🤖**
  * Because we are predicting a continuous number (price), this is a **Regression** problem.
  * We’ll start with a straightforward linear model to set a baseline score. Once that’s locked in, we’ll unleash heavy-hitting gradient-boosted trees like **XGBoost** and **LightGBM** to handle the complex, non-linear relationships in the data.

* **4. Keeping Evaluation Realistic 📊**
  * We will test our models using **RMSLE** (Root Mean Squared Log Error). Why? Because a \$20,000 mistake on a \$100,000 starter home is a disaster, but a \$20,000 mistake on a \$2,000,000 mansion is practically a rounding error. Log error keeps our penalties fair across all price brackets.

---

## **💡 Coding Standards (No Sloppy Notebooks)**

We are writing code that looks like it belongs in a production environment, not just a sandbox:

* **Readable & Modular:** No giant blocks of messy code. We’ll write clean, reusable python functions with clear descriptions.
* **Bulletproof Integrity:** We will explicitly validate our data shapes and types using `pandas` before passing anything to our machine learning models. 
* **Scalable Thinking:** The logic we write for this dataset will be clean enough to easily scale up to enterprise-level data down the road.

### 🚀 Automated Data Ingestion

To ensure maximum reproducibility and maintain clean versioning, we pull the dataset directly using Kaggle's tools. This automated process fetches the raw housing feature records—tracking structural properties, location metrics, and sales history—directly into our environment.

* **Dataset Credit:** Vedat Gül via Kaggle (*House Prices Prediction / Advanced Regression Techniques*).
* **Source Notebook/Data:** [Kaggle Notebook Link](https://www.kaggle.com/datasets/fratzcan/usa-house-prices)

### 📥 Loading the Dataset and Libraries

Before we start, we need to install the necessary Python libraries and **load the dataset**.


In [1]:
# Install the required libraries
%pip install kagglehub pandas numpy matplotlib seaborn scikit-learn -q
%pip install ucimlrepo -q

# Install and update the watermark package to display environment and library version information
%pip install -q -U watermark

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


### 📥 Loading the Dataset and Libraries

Before we start, we need to install the necessary Python libraries and **load the dataset**.


In [2]:
# IMPORT REQUIRED MODULES - Data manipulation and visualization
import os
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Pre-Processing and Machine Learning
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

c:\Users\LarTI\OneDrive\Desktop\Projects\House_Prices_Prediction\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
# Load the watermark extension to log the environment state
%reload_ext watermark

# Display professional metadata tracking our data engineering stack
%watermark -a "Maykon - 🏡 The House Price Engine" -d -u -v -p pandas,numpy,matplotlib,seaborn,scikit-learn,xgboost,lightgbm

Author: Maykon - 🏡 The House Price Engine

Last updated: 2026-06-11

Python implementation: CPython
Python version       : 3.13.7
IPython version      : 9.14.1

pandas      : 3.0.3
numpy       : 2.4.6
matplotlib  : 3.10.9
seaborn     : 0.13.2
scikit-learn: 1.9.0
xgboost     : unknown
lightgbm    : unknown



In [8]:
# Download latest version
path = kagglehub.dataset_download("fratzcan/usa-house-prices")

print("📦 Path to dataset files:", path)

📦 Path to dataset files: C:\Users\LarTI\.cache\kagglehub\datasets\fratzcan\usa-house-prices\versions\1


In [9]:
# --- LOCATING AND READING THE CSV ---
# List out all files inside the downloaded repository path to spot the target file
all_files = os.listdir(path)
print("📂 Files discovered in directory:", all_files)

# Filter out all CSV files dynamically
csv_files = [file for file in all_files if file.endswith('.csv')]

if len(csv_files) == 0:
    raise FileNotFoundError("❌ Critical Error: No CSV files found in the downloaded folder!")
else:
    # Grab the primary CSV file found
    csv_filename = csv_files[0]
    full_csv_path = os.path.join(path, csv_filename)
    print(f"🎯 Target CSV located: {csv_filename}")

📂 Files discovered in directory: ['USA Housing Dataset.csv']
🎯 Target CSV located: USA Housing Dataset.csv


In [11]:
# Ingest the dataset into a pandas DataFrame
df = pd.read_csv(full_csv_path)
print(f"✅ Dataset successfully loaded! Shape: {df.shape[0]} rows, {df.shape[1]} columns.")

✅ Dataset successfully loaded! Shape: 4140 rows, 18 columns.


In [12]:
# Display the first 5 records to see our column properties and labels
df.head()

,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,sqft_above,sqft_basement,yr_built,yr_renovated,street,city,statezip,country
0,2014-05-09 00:00:00,376000.0,3.0,2.00,1340,1384,3.0,0,0,3,1340,0,2008,0,9245-9249 Fremont Ave N,Seattle,WA 98103,USA
1,2014-05-09 00:00:00,800000.0,4.0,3.25,3540,159430,2.0,0,0,3,3540,0,2007,0,33001 NE 24th St,Carnation,WA 98014,USA
2,2014-05-09 00:00:00,2238888.0,5.0,6.50,7270,130017,2.0,0,0,3,6420,850,2010,0,7070 270th Pl SE,Issaquah,WA 98029,USA
3,2014-05-09 00:00:00,324000.0,3.0,2.25,998,904,2.0,0,0,3,798,200,2007,0,820 NW 95th St,Seattle,WA 98117,USA
4,2014-05-10 00:00:00,549900.0,5.0,2.75,3060,7015,1.0,0,0,5,1600,1460,1979,0,10834 31st Ave SW,Seattle,WA 98146,USA


In [13]:
df.sample(15) # Random 15 rows

,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,sqft_above,sqft_basement,yr_built,yr_renovated,street,city,statezip,country
1641,2014-06-09 00:00:00,390000.0,4.0,1.75,2700,7875,1.5,0,0,4,2700,0,1968,0,16518-16526 147th Ave SE,Renton,WA 98058,USA
2779,2014-06-24 00:00:00,850000.0,4.0,3.50,3920,37122,2.0,0,0,3,3920,0,1996,0,2515 263rd Ct NE,Redmond,WA 98053,USA
2213,2014-06-17 00:00:00,472000.0,3.0,2.50,1180,1262,3.0,0,0,3,1180,0,2010,0,2050 14th Ave W,Seattle,WA 98119,USA
3068,2014-06-27 00:00:00,140000.0,3.0,1.00,1060,7473,1.0,0,0,3,1060,0,1959,1989,25826 19th Ave S,Des Moines,WA 98198,USA
322,2014-05-16 00:00:00,252700.0,2.0,1.50,1070,9643,1.0,0,0,3,1070,0,1985,0,13105 SE 277th Pl,Kent,WA 98030,USA
1574,2014-06-06 00:00:00,510000.0,3.0,1.75,1480,7040,1.0,0,0,3,1480,0,1974,0,7517 128th Pl NE,Kirkland,WA 98033,USA
2090,2014-06-16 00:00:00,963000.0,4.0,3.50,3280,6603,2.0,0,0,3,3280,0,2007,0,16030 SE 45th Pl,Bellevue,WA 98006,USA
1947,2014-06-12 00:00:00,498500.0,5.0,2.75,2990,7420,2.0,0,0,3,2990,0,1996,0,4304 NE 7th St,Renton,WA 98059,USA
3015,2014-06-26 00:00:00,243800.0,3.0,1.00,1140,27760,1.0,0,0,4,1140,0,1981,0,27529 SE High Point Way,Issaquah,WA 98027,USA
1430,2014-06-04 00:00:00,532000.0,4.0,1.75,2020,7029,1.0,0,0,4,1430,590,1979,0,10916 159th Ave NE,Redmond,WA 98052,USA


In [14]:
df.tail() #Displays the last 5 rows of the DataFrame df.

,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,sqft_above,sqft_basement,yr_built,yr_renovated,street,city,statezip,country
4135,2014-07-09 00:00:00,308166.666667,3.0,1.75,1510,6360,1.0,0,0,4,1510,0,1954,1979,501 N 143rd St,Seattle,WA 98133,USA
4136,2014-07-09 00:00:00,534333.333333,3.0,2.50,1460,7573,2.0,0,0,3,1460,0,1983,2009,14855 SE 10th Pl,Bellevue,WA 98007,USA
4137,2014-07-09 00:00:00,416904.166667,3.0,2.50,3010,7014,2.0,0,0,3,3010,0,2009,0,759 Ilwaco Pl NE,Renton,WA 98059,USA
4138,2014-07-10 00:00:00,203400.000000,4.0,2.00,2090,6630,1.0,0,0,3,1070,1020,1974,0,5148 S Creston St,Seattle,WA 98178,USA
4139,2014-07-10 00:00:00,220600.000000,3.0,2.50,1490,8102,2.0,0,0,4,1490,0,1990,0,18717 SE 258th St,Covington,WA 98042,USA


#### 🔎📊 Exploratory Data Analysis (EDA)

The investigation phase begins here. Before making any changes or feeding numbers into an algorithm, we dive deep into the data to uncover underlying market patterns, spot anomalies, and catch structural flaws 💡. 

Using statistical measures and targeted data visualizations 📈 (like plotting the distribution of house prices to check for luxury outliers or mapping correlations between square footage and final sale value), we analyze how different home characteristics behave. This critical health check ensures we understand exactly what the data is telling us before we write a single line of feature engineering or modeling code 🛠️.

In [15]:
# Information about the dataframe
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4140 entries, 0 to 4139
Data columns (total 18 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   date           4140 non-null   str    
 1   price          4140 non-null   float64
 2   bedrooms       4140 non-null   float64
 3   bathrooms      4140 non-null   float64
 4   sqft_living    4140 non-null   int64  
 5   sqft_lot       4140 non-null   int64  
 6   floors         4140 non-null   float64
 7   waterfront     4140 non-null   int64  
 8   view           4140 non-null   int64  
 9   condition      4140 non-null   int64  
 10  sqft_above     4140 non-null   int64  
 11  sqft_basement  4140 non-null   int64  
 12  yr_built       4140 non-null   int64  
 13  yr_renovated   4140 non-null   int64  
 14  street         4140 non-null   str    
 15  city           4140 non-null   str    
 16  statezip       4140 non-null   str    
 17  country        4140 non-null   str    
dtypes: float64(4), int6

In [16]:
df.describe(include='all').T  # Generates descriptive statistics for all numeric columns in your DataFrame.

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
date,4140,68,2014-06-23 00:00:00,142,NaN,NaN,NaN,NaN,NaN,NaN,NaN
price,4140.0,NaN,NaN,NaN,553062.877289,583686.452245,0.0,320000.0,460000.0,659125.0,26590000.0
bedrooms,4140.0,NaN,NaN,NaN,3.400483,0.903939,0.0,3.0,3.0,4.0,8.0
bathrooms,4140.0,NaN,NaN,NaN,2.163043,0.784733,0.0,1.75,2.25,2.5,6.75
sqft_living,4140.0,NaN,NaN,NaN,2143.638889,957.481621,370.0,1470.0,1980.0,2620.0,10040.0
sqft_lot,4140.0,NaN,NaN,NaN,14697.638164,35876.838123,638.0,5000.0,7676.0,11000.0,1074218.0
floors,4140.0,NaN,NaN,NaN,1.51413,0.534941,1.0,1.0,1.5,2.0,3.5
waterfront,4140.0,NaN,NaN,NaN,0.007488,0.086219,0.0,0.0,0.0,0.0,1.0
view,4140.0,NaN,NaN,NaN,0.246618,0.790619,0.0,0.0,0.0,0.0,4.0
condition,4140.0,NaN,NaN,NaN,3.452415,0.678533,1.0,3.0,3.0,4.0,5.0


##### 🔍 Checking for Missing Data (Null Value Audit)

In [21]:
df.isna().sum()  # You can also use df.isnull().sum() — both do the same thing

date             0
price            0
bedrooms         0
bathrooms        0
sqft_living      0
sqft_lot         0
floors           0
waterfront       0
view             0
condition        0
sqft_above       0
sqft_basement    0
yr_built         0
yr_renovated     0
street           0
city             0
statezip         0
country          0
dtype: int64

In [18]:
# Calculate absolute counts and percentages of missing data per column
missing_counts = df.isna().sum()
missing_percentages = (df.isna().sum() / len(df)) * 100

# Combine the results into a clean summary table
missing_data_summary = pd.DataFrame({
    'Total Missing': missing_counts,
    'Percentage (%)': missing_percentages
})

# Filter out the columns that are 100% clean so we can focus on the trouble areas
trouble_columns = missing_data_summary[missing_data_summary['Total Missing'] > 0].sort_values(by='Total Missing', ascending=False)

if trouble_columns.empty:
    print("✨ Clean Data Check: No missing values found anywhere in the dataset!")
else:
    print(f"⚠️ Found {len(trouble_columns)} columns with missing data.")
    print(trouble_columns)

✨ Clean Data Check: No missing values found anywhere in the dataset!
